# 06 ─ Survival Analysis for ESP Failure Prediction

This notebook applies survival analysis to predict the **time-to-failure** distribution for Electric Submersible Pumps (ESPs).

**Models:**
1. **Cox Proportional Hazards (Cox PH)** ─ Semi-parametric, no distributional assumption on baseline hazard
2. **Weibull Accelerated Failure Time (AFT)** ─ Parametric, directly interpretable scale/shape

**Outputs:**
- Hazard function h(t|x): instantaneous failure rate
- Survival function S(t|x): probability of surviving beyond time t
- Median survival time and failure probability within a horizon
- Hazard ratios: how each sensor affects failure risk

In [ ]:
# -- Cell 1: Environment setup -------------------------------------------------
import sys, os
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # Clone the full repo (Colab only copies this notebook by default)
    if not os.path.exists('src'):
        !git clone https://github.com/WickDager/esp-predictive-maintenance.git
        %cd esp-predictive-maintenance

    !pip install -q torch torchvision tqdm scikit-learn

    # Use LOCAL storage for training (reliable, no Drive I/O issues)
    # At the end, a final cell will copy everything to Google Drive
    SAVE_DIR = '/content/checkpoints'
else:
    SAVE_DIR = 'checkpoints'

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs('results', exist_ok=True)
print(f'Checkpoints will be saved to: {SAVE_DIR}')

import torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


## 1. Generate Data

We use synthetic data with multiple wells, each having a different failure mode.

In [ ]:
# Generate multiple wells with varying failure modes
# CHOOSE: synthetic or real pump sensor data
USE_REAL_DATA = False   # Set True to load CSV from Drive

if USE_REAL_DATA:
    DRIVE_CSV_PATH = '/content/drive/MyDrive/pump_sensor.csv'
    print(f'Loading real data from {DRIVE_CSV_PATH}...')
    df = pd.read_csv(DRIVE_CSV_PATH, parse_dates=['timestamp'])
    df = df.sort_values('timestamp').reset_index(drop=True)
    SENSOR_COLS = sorted([c for c in df.columns if c.startswith('sensor_')])
    df[SENSOR_COLS] = df[SENSOR_COLS].ffill().bfill()
    status_map = {'NORMAL': 0, 'RECOVERING': 0, 'BROKEN': 1}
    df['failure'] = df['machine_status'].map(status_map).fillna(0).astype(int)
    df['failure_mode'] = 'unknown'  # real data has no failure mode labels
else:
    from src.data.synthetic_generator import generate_esp_dataset, SYNTHETIC_SENSOR_COLS
    df = generate_esp_dataset(
        n_wells=50, timesteps_per_well=2000,
        failure_prob=0.7, random_seed=42,
    )
    df['failure'] = (df['machine_status'] == 'BROKEN').astype(int)
    SENSOR_COLS = SYNTHETIC_SENSOR_COLS

print(f"Total wells: {df['well_id'].nunique()}")
print(f"Failure distribution:")
print(df.groupby(['failure_mode', 'machine_status']).size())

## 2. Prepare Survival Data

Survival analysis requires one row per unit (well), not per timestep:
- `duration`: time-to-event (or censoring time)
- `event`: 1 if failure observed, 0 if censored
- `covariates`: summary statistics of sensor signals

In [ ]:
surv_df = prepare_survival_dataframe(
    df,
    sensor_cols=SYNTHETIC_SENSOR_COLS,
    time_col='rul',
    event_col='failure',
    groupby_col='well_id',
    agg_window=100,  # Use first 100 timesteps for covariates
)

print(f"Survival dataset: {surv_df.shape}")
print(f"Events (failures): {surv_df['event'].sum()}")
print(f"Censored: {(surv_df['event'] == 0).sum()}")
print(f"\nDuration statistics:")
print(surv_df['duration'].describe())

## 3. Cox Proportional Hazards Model

In [ ]:
# Select a subset of covariates for interpretability
covariate_cols = [c for c in surv_df.columns if c.endswith('_mean') or c.endswith('_trend')]
use_cols = ['duration', 'event'] + covariate_cols[:10]  # Limit to avoid overfitting on synthetic data
surv_df_model = surv_df[use_cols].dropna()

print(f"Modeling with {len(use_cols)-2} covariates, {len(surv_df_model)} units")

In [ ]:
# Fit Cox PH model
cox = CoxPHModel(penalizer=0.5, l1_ratio=0.0)
cox.fit(surv_df_model, duration_col='duration', event_col='event')

print("\nCox PH Model Summary:")
cox.print_summary()

In [ ]:
# Hazard ratios
hr_df = cox.get_hazard_ratios()
print("\nHazard Ratios (exp(coef)):")
print(hr_df.head(10))

# Plot hazard ratios
fig, ax = plt.subplots(figsize=(8, max(4, len(hr_df) * 0.3)))
hr_sorted = hr_df.sort_values('exp(coef)')
y_pos = range(len(hr_sorted))
ax.barh(y_pos, hr_sorted['exp(coef)'], color='#378ADD', alpha=0.8)
ax.set_yticks(y_pos)
ax.set_yticklabels(hr_sorted.index, fontsize=8)
ax.axvline(1.0, color='#E24B4A', linestyle='--', alpha=0.7, label='HR = 1.0 (no effect)')
ax.set_xlabel('Hazard Ratio (exp())', fontsize=11)
ax.set_title('Cox PH ─ Hazard Ratios for ESP Failure', fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Concordance index
c_index = cox.concordance_index(surv_df_model)
print(f"Concordance Index: {c_index:.4f}")
print("(0.5 = random, 1.0 = perfect discrimination)")

In [ ]:
# Plot survival curves for different covariate profiles
# Create 3 profiles: low-risk, medium-risk, high-risk
covariate_features = [c for c in surv_df_model.columns if c not in ['duration', 'event']]

low_risk = surv_df_model[covariate_features].mean().to_frame().T
high_risk = surv_df_model[covariate_features].mean().to_frame().T

# Adjust a key feature to create contrast
vib_col = [c for c in covariate_features if 'vibration' in c]
if vib_col:
    low_risk[vib_col[0]] = surv_df_model[vib_col[0]].quantile(0.1)
    high_risk[vib_col[0]] = surv_df_model[vib_col[0]].quantile(0.9)

medium_risk = (low_risk + high_risk) / 2

sample_profiles = pd.concat([low_risk, medium_risk, high_risk], ignore_index=True)
sample_profiles.index = ['Low Risk', 'Medium Risk', 'High Risk']

fig = plt.figure(figsize=(10, 5))
plot_survival_curves(
    cox, sample_profiles,
    labels=['Low Risk', 'Medium Risk', 'High Risk'],
    title='Cox PH ─ Predicted Survival Functions by Risk Profile'
)
plt.show()

## 4. Weibull AFT Model

In [ ]:
# Fit Weibull AFT model
weibull = WeibullAFTModel()
weibull.fit(surv_df_model, duration_col='duration', event_col='event')

print("Weibull AFT Model Summary:")
weibull.print_summary()

In [ ]:
# Compare survival curves from both models
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Cox PH
sf_cox = cox.model.predict_survival_function(sample_profiles)
for col in sf_cox.columns:
    axes[0].plot(sf_cox.index, sf_cox[col], linewidth=2, label=col)
axes[0].axhline(0.5, color='gray', linestyle='--', alpha=0.5)
axes[0].set_xlabel('Time')
axes[0].set_ylabel('S(t|x)')
axes[0].set_title('Cox PH Survival Curves')
axes[0].legend()

# Weibull AFT
sf_weibull = weibull.model.predict_survival_function(sample_profiles)
for col in sf_weibull.columns:
    axes[1].plot(sf_weibull.index, sf_weibull[col], linewidth=2, label=col)
axes[1].axhline(0.5, color='gray', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Time')
axes[1].set_ylabel('S(t|x)')
axes[1].set_title('Weibull AFT Survival Curves')
axes[1].legend()

plt.tight_layout()
plt.show()

## 5. Failure Probability Forecasting

Compute the probability of failure within a specific time horizon (e.g., next 500 timesteps).

In [ ]:
horizon = 500
failure_probs = cox.predict_failure_probability(sample_profiles, horizon=horizon)

print(f"\nProbability of failure within {horizon} timesteps:")
for label, prob in zip(sample_profiles.index, failure_probs):
    print(f"  {label:15s}: {prob:.1%}")

# Bar chart
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(sample_profiles.index, failure_probs, color=['#1D9E75', '#EF9F27', '#E24B4A'])
ax.set_ylabel(f'P(Failure within {horizon} timesteps | x)', fontsize=10)
ax.set_title('Cox PH ─ Failure Probability by Risk Profile', fontsize=12)
ax.set_ylim(0, 1)
for i, (label, prob) in enumerate(zip(sample_profiles.index, failure_probs)):
    ax.text(i, prob + 0.02, f'{prob:.1%}', ha='center', fontsize=10)
plt.tight_layout()
plt.show()

## 6. Practical Usage for ESP Maintenance

Survival analysis provides **actionable maintenance insights**:

| Question | Model Answer |
|----------|-------------|
| When is this pump likely to fail? | Median survival time from Weibull AFT |
| What's the probability of failure in 30 days? | `predict_failure_probability(horizon=30*24*60)` |
| Which sensors indicate highest risk? | Hazard ratios from Cox PH |
| Should we schedule maintenance now? | Compare S(t|x) for different maintenance horizons |

**Integration with the full pipeline:**
- Cox PH hazard scores can be used as additional features for the RUL predictor
- Survival curves complement the MC Dropout uncertainty from the neural models
- Weibull AFT expected failure times can serve as baseline targets

## 7. Summary

Survival analysis provides a **statistically rigorous** approach to failure prediction:

- **Cox PH**: Best for understanding which factors drive failure risk (interpretable hazard ratios)
- **Weibull AFT**: Best for direct lifetime estimation (expected failure time)
- Both models handle **censored data** naturally (wells that haven't failed yet)
- Complement deep learning approaches with **transparent, auditable** predictions

**Next step:** See the model evaluation notebook (`07_Model_Evaluation_and_SHAP.ipynb`) for a full comparison of all models.

In [ ]:
# -- Save results to Google Drive -------------------------------------------
import sys, os, shutil

if 'google.colab' not in sys.modules:
    print('Not running in Colab. No Drive sync needed.')
else:
    from google.colab import drive
    drive.mount('/content/drive')

    DRIVE_DIR = '/content/drive/MyDrive/esp_pm'
    os.makedirs(f'{DRIVE_DIR}/checkpoints', exist_ok=True)
    os.makedirs(f'{DRIVE_DIR}/results', exist_ok=True)

    # Copy checkpoints
    if os.path.exists('checkpoints'):
        for f in os.listdir('checkpoints'):
            src = os.path.join('checkpoints', f)
            dst = f'{DRIVE_DIR}/checkpoints/{f}'
            shutil.copy2(src, dst)
            print(f'  Copied: checkpoints/{f}')

    # Copy results
    if os.path.exists('results'):
        for f in os.listdir('results'):
            src = os.path.join('results', f)
            dst = f'{DRIVE_DIR}/results/{f}'
            shutil.copy2(src, dst)
            print(f'  Copied: results/{f}')

    # Copy training log
    if os.path.exists(f'{SAVE_DIR}/training_log.csv'):
        shutil.copy2(f'{SAVE_DIR}/training_log.csv', f'{DRIVE_DIR}/checkpoints/training_log.csv')
        print(f'  Copied: training_log.csv')

    print(f'\nAll files synced to: {DRIVE_DIR}')
